# Predição do modelo de classificação de coral-sol, nas provas de foto (1, 2 e 3)

Autor:  Viviane da Rosa Sommer

Atualizado: 27/12/2024

## Objetivo:
Notebook destinado à predição das imagens das provas de foto utilizando o modelo V1 de classificação de coral-sol, com o objetivo de analisar seu desempenho.

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless
!pip install --upgrade tensorflow
!pip install --upgrade keras
import keras
import glob
import numpy as np
import matplotlib.pyplot as plt
from keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

## Declaração de Constantes e Modelos

In [ ]:
model = modelo = keras.models.load_model('best_weights.keras')
batch_size = 32

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Read an image from a file and convert it to a NumPy array.

    Args:
        filename (str): The path to the image file.

    Returns:
        np.ndarray: The image as a NumPy array.
    """
    image = load_img(filename)
    image = img_to_array(image)
    height, width, _ = image.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_image = image[top:bottom, :]

    return cropped_image
    

def pad_sub_image(sub_image: np.ndarray, size: int) -> np.ndarray:
    """
    Pad a sub-image to ensure it is of the given size.

    Args:
        sub_image (np.ndarray): The original sub-image.
        size (int): The desired size of the sub-image.

    Returns:
        np.ndarray: The padded sub-image.
    """
    padded_image = np.zeros((size, size, 3), dtype=sub_image.dtype)
    padded_image[:sub_image.shape[0], :sub_image.shape[1], :] = sub_image
    return padded_image
    

def divide_image(image: np.ndarray, size: int) -> tuple[list, list]:
    """
    Divide an image into sub-images of a given size.

    Args:
        image (np.ndarray): The original image.
        size (int): The size of each sub-image.

    Returns:
        tuple[list, list]: List of sub-images and list of (x, y) coordinates of the sub-images.
    """
    sub_images = []
    coordinates = []
    height, width, _ = image.shape
    for y in range(0, height, size):
        for x in range(0, width, size):
            sub_image = image[y:min(y+size, height), x:min(x+size, width)]
            sub_image = pad_sub_image(sub_image, size)
            sub_images.append(sub_image)
            coordinates.append((x, y))
    return sub_images, coordinates
    

def batch_predict(model: keras.Model, data: list, batch_size: int) -> np.ndarray:
    """
    Predict in batches to avoid memory issues.

    Args:
        model (keras.Model): The trained model.
        data (list): The data to predict.
        batch_size (int): The size of each batch.

    Returns:
        np.ndarray: The predictions.
    """
    predictions = []
    for i in range(0, len(data), batch_size):
        batch = np.array(data[i:i+batch_size])
        batch_predictions = model.predict(batch, verbose=0)
        predictions.extend(batch_predictions)
    return np.array(predictions)
    

def plot_results(image: np.ndarray, results: np.ndarray, coordinates: list, 
                 size: int, title: str) -> None:
    """
    Plot the image and mark sub-images as positive or negative.

    Args:
        image (np.ndarray): The original image.
        results (np.ndarray): The binary results for each sub-image.
        coordinates (list): List of (x, y) coordinates of the sub-images.
        size (int): The size of each sub-image.
    """
    fig, ax = plt.subplots(1)
    ax.imshow(image.astype(np.uint8))
    for idx, (x, y) in enumerate(coordinates):
        if not results[idx]:
            rect = plt.Rectangle((x, y), size, size, linewidth=2, edgecolor='blue', facecolor='none')
            ax.add_patch(rect)

    for idx, (x, y) in enumerate(coordinates):
        if results[idx]:
            rect = plt.Rectangle((x, y), size, size, linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
    plt.title(title)
    plt.show()


def evaluate_classification(true_labels: list[str], predicted_labels: list[str]) -> None:
    """
    Evaluates the classification performance by generating a confusion matrix 
    and a classification report.

    Parameters:
    - true_labels (list[str]): List of true labels (strings: 'Positive' or 'Negative').
    - predicted_labels (list[str]): List of predicted labels (strings: 'Positive' or 'Negative').

    Returns:
    - None: Results are displayed directly through a confusion matrix plot and a printed classification report.
    """
    
    true_labels_binary = [1 if label == 'Positive' else 0 for label in true_labels]
    predicted_labels_binary = [1 if label == 'Positive' else 0 for label in predicted_labels]

    conf_matrix = confusion_matrix(true_labels_binary, predicted_labels_binary)
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Negative', 'Positive'])
    disp.plot(cmap=plt.cm.Blues)
    plt.show()

    report = classification_report(true_labels_binary, predicted_labels_binary, target_names=['Negative', 'Positive'])
    print(report)

## Processamento das imagens - OS 6000696049

In [ ]:
true_labels = []
predicted_labels = []
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto _6000696049/**"
for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**"):
    try:
        image = read_image(filename)
        if image is None:
            continue

        sub_images, coordinates = divide_image(image, 128)
        sub_images = np.array(sub_images) / 255.0

        results = batch_predict(model, sub_images, batch_size)
        binary_results = results >= 0.5

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        predicted_label = 'Positive' if np.any(binary_results) else 'Negative'

        true_labels.append(true_label)
        predicted_labels.append(predicted_label)

        prediction_status = "Positive" if np.any(binary_results) else "Negative"
        title = f"Model prediction = {prediction_status} \nTrue label = {true_label}"
        plot_results(image, binary_results, coordinates, 128, title)

    except Exception as e:
        print(f"Error processing file {filename}: {e}")

## Matriz de confusão - OS 6000696049

In [ ]:
evaluate_classification(true_labels, predicted_labels)

## Processamento das imagens - OS 6000504195

In [ ]:
true_labels = []
predicted_labels = []
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto_6000504195/**"
for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**"):
    try:
        image = read_image(filename)
        if image is None:
            continue

        sub_images, coordinates = divide_image(image, 128)
        sub_images = np.array(sub_images) / 255.0

        results = batch_predict(model, sub_images, batch_size)
        binary_results = results >= 0.5

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        predicted_label = 'Positive' if np.any(binary_results) else 'Negative'

        true_labels.append(true_label)
        predicted_labels.append(predicted_label)

        prediction_status = "Positive" if np.any(binary_results) else "Negative"
        title = f"Model prediction = {prediction_status} \nTrue label = {true_label}"
        plot_results(image, binary_results, coordinates, 128, title)

    except Exception as e:
        print(f"Error processing file {filename}: {e}")

## Matriz de confusão - OS 6000504195

In [ ]:
evaluate_classification(true_labels, predicted_labels)

## Processamento das imagens - OS 6000701944

In [ ]:
true_labels = []
predicted_labels = []
INPUT_DIRECTORY_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de Foto_6000701944/**"
for filename in glob.glob(f"{INPUT_DIRECTORY_IMAGES}/**"):
    try:
        image = read_image(filename)
        if image is None:
            continue

        sub_images, coordinates = divide_image(image, 128)
        sub_images = np.array(sub_images) / 255.0

        results = batch_predict(model, sub_images, batch_size)
        binary_results = results >= 0.5

        true_label = 'Positive' if "POSITIVA" in filename.upper() else 'Negative'
        predicted_label = 'Positive' if np.any(binary_results) else 'Negative'

        true_labels.append(true_label)
        predicted_labels.append(predicted_label)

        prediction_status = "Positive" if np.any(binary_results) else "Negative"
        title = f"Model prediction = {prediction_status} \nTrue label = {true_label}"
        plot_results(image, binary_results, coordinates, 128, title)

    except Exception as e:
        print(f"Error processing file {filename}: {e}")

## Matriz de confusão - OS 6000701944

In [ ]:
evaluate_classification(true_labels, predicted_labels)

In [ ]:
!jupyter nbconvert --to html prediction_prova_foto.ipynb